In [1]:
!pip uninstall -y unsloth unsloth_zoo trl peft bitsandbytes transformers accelerate

!pip install --no-cache-dir bitsandbytes accelerate datasets peft

!pip install --no-cache-dir transformers==5.0.0

!pip install --no-cache-dir trl==1.5.1

!pip install --no-deps unsloth_zoo
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 287.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 384.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 413.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 270.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 245.0 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take i

In [2]:
import os
import glob
import pandas as pd
from datasets import Dataset

# 1. Define the directory path from your Kaggle input
dataset_dir = "/kaggle/input/datasets/adhamashraf202200953/technical-parsed-questions/"

# 2. Find all files that contain '_essay_variants.csv'
file_pattern = os.path.join(dataset_dir, "*_essay_variants.csv")
essay_files = glob.glob(file_pattern)

print(f"Found {len(essay_files)} essay variant files to concatenate:")
for file in essay_files:
    print(f" - {os.path.basename(file)}")

# 3. Read and concatenate all found csv files
df_list = [pd.read_csv(file) for file in essay_files]
combined_df = pd.concat(df_list, ignore_index=True)
print(f"\nTotal records loaded after concatenation: {len(combined_df)}")

# 4. Map the flat tabular dataset into a standardized chat structure
def format_row_to_chat(row):
    # Dynamically extract all populated yes/no columns for this specific row
    yn_cols = [
        str(row[c]).strip() 
        for c in row.index 
        if c.startswith('yes_no_questions/') and pd.notna(row[c]) and str(row[c]).strip() != ""
    ]
    
    # Format the list of grading questions into bullet points
    formatted_questions = "\n".join([f"- {q}" for q in yn_cols])
    
    # Structure the conversations matching LLM instruction setups
    messages = [
        {
            "role": "system", 
            "content": "You are an automated technical assessment expert. Analyze the provided Question and Model Answer to produce a comprehensive Rubric Description and list out exact binary Yes/No questions to grade candidate submissions."
        },
        {
            "role": "user", 
            "content": f"Context: {row.get('context', 'N/A')}\nQuestion: {row.get('question', 'N/A')}\nModel Answer: {row.get('model_answer', 'N/A')}"
        },
        {
            "role": "assistant", 
            "content": f"### RUBRIC DESCRIPTION\n{row.get('rubric_description', 'N/A')}\n\n### YES/NO QUESTIONS\n{formatted_questions}"
        }
    ]
    return {"messages": messages}

# Format and convert to Hugging Face Dataset format
formatted_data = combined_df.apply(format_row_to_chat, axis=1).tolist()
dataset = Dataset.from_list(formatted_data)
print("Dataset successfully mapped to Chat templates.")

Found 4 essay variant files to concatenate:
 - SWE_Abstract_Theory_essay_variants.csv
 - DSA_Applied_Enginering_essay_variants.csv
 - DSA_Abstract_Theory_essay_variants.csv
 - ML_DS_Abstract_Theory_essay_variants.csv

Total records loaded after concatenation: 2928
Dataset successfully mapped to Chat templates.


In [3]:
# !pip install trl

In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 
dtype = None 
load_in_4bit = True 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.10: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.
Unsloth 2026.5.10 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [5]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"},
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/2928 [00:00<?, ? examples/s]

In [6]:
import os
import shutil
import torch
from trl import SFTTrainer, SFTConfig

if os.path.exists("/kaggle/working/unsloth_compiled_cache"):
    shutil.rmtree("/kaggle/working/unsloth_compiled_cache")

torch.cuda.empty_cache()

training_config = SFTConfig(
    per_device_train_batch_size = 2,      
    gradient_accumulation_steps = 4,     
    warmup_steps = 5,
    num_train_epochs = 1,                 
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,                   
    optim = "adamw_8bit",                
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
    report_to = "none",
    
    dataset_text_field = "text",
    packing = True,                     
    max_seq_length = 2048,                
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_num_proc = 2,
    args = training_config,
)

trainer.model.gradient_checkpointing_enable()

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/2928 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=8):   0%|          | 0/2928 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,302 | Num Epochs = 1 | Total steps = 163
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.139164
2,2.258982
3,2.266608
4,2.073644
5,2.040528
6,1.890285
7,1.880631
8,1.813902
9,1.788017
10,1.719867


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-163/tokenizer_config.json.


In [7]:
# Transition model to inference speeds
FastLanguageModel.for_inference(model) 

test_messages = [
    {"role": "system", "content": "You are an automated technical assessment expert..."},
    {"role": "user", "content": "Context: Performance optimizations on system processing loops.\nQuestion: Explain the benefit of loop unrolling.\nModel Answer: Loop unrolling increases performance by eliminating loop overhead instructions and optimizing instruction-level parallelism."}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


system

You are an automated technical assessment expert...user

Context: Performance optimizations on system processing loops.
Question: Explain the benefit of loop unrolling.
Model Answer: Loop unrolling increases performance by eliminating loop overhead instructions and optimizing instruction-level parallelism.assistant

Context: Performance optimizations on system processing loops.
Question: Explain the benefit of loop unrolling.
Model Answer: Loop unrolling benefits system processing loops by increasing instruction-level parallelism, reducing branch prediction mispredictions, and improving cache locality. This results in improved performance, reduced power consumption, and increased throughput.


In [8]:
model.save_pretrained_merged("technical_rubric_generator", tokenizer, save_method = "lora")
print("Model adapters saved locally to the Kaggle working directory.")

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in technical_rubric_generator/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:16<00:00,  8.38s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:42<00:00, 21.25s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/technical_rubric_generator`
Model adapters saved locally to the Kaggle working directory.
